# ParkCast SF — Active Street-Use Permits Feature

A block under an active excavation, emergency, or street-occupancy permit is physically blocked — meters don't run, paid-sessions drop to near zero, and the block is *not* empty, it's *closed*. Without this flag the model sees a permit-closed block at 0% occupancy and learns that's a normal daytime condition for that block.

For each block we compute:
- `permit_active` — 1 if any permit type in `BLOCKING_TYPES` is active on this (cnn, date).
- `permit_count_30d` — rolling count of active blocking-permit days in the past 30, as a "this block is chronically disrupted" signal.

**Output:** `data/block_permit_features.csv` keyed on `(cnn, date)`.

## Imports

In [1]:
import os
import numpy as np
import pandas as pd

## Configuration

In [2]:
PROJECT_DIR = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(PROJECT_DIR, "data")

SRC = os.path.join(DATA_DIR, "street_use_permits.csv")
OUT = os.path.join(DATA_DIR, "block_permit_features.csv")

# Permit types that physically block parking. Banners/Display/Sidewalk-only
# permits do not. Excavation digs up roadway; ExcStreet is street-excavation;
# TempOccup is temporary street occupancy (contractor trucks, cranes);
# Emergency is utility emergency work. Valid values per DataSF schema.
BLOCKING_TYPES = ["Excavation", "ExcStreet", "TempOccup", "Emergency"]

# Look 14 months back and 1 month forward to cover the training window
# (2025-04-01 through 2026-04-12) plus a buffer for near-future inference.
WINDOW_START = pd.Timestamp("2025-03-01")
WINDOW_END   = pd.Timestamp("2026-05-01")

## `load_permits()`

Filter to blocking permit types and permits whose date span intersects our training window. Blockfaces are identified by `cnn` — the DPW centerline ID that the preprocess pipeline already joins on.

In [3]:
def load_permits():
    cols = ["cnn", "permit_type", "permit_start_date", "permit_end_date", "status"]
    df = pd.read_csv(SRC, usecols=cols, low_memory=False)
    print(f"  Raw permits: {len(df):,}")

    df["permit_start_date"] = pd.to_datetime(df["permit_start_date"], errors="coerce")
    df["permit_end_date"]   = pd.to_datetime(df["permit_end_date"],   errors="coerce")
    df = df.dropna(subset=["cnn", "permit_start_date", "permit_end_date",
                          "permit_type"])
    df["cnn"] = df["cnn"].astype(np.int64)

    # Keep only the permit types that actually occupy roadway.
    before = len(df)
    df = df[df["permit_type"].isin(BLOCKING_TYPES)].copy()
    print(f"  After blocking-type filter: {len(df):,}  (dropped {before - len(df):,})")

    # Intersect with our training window.
    df = df[(df["permit_end_date"] >= WINDOW_START) &
            (df["permit_start_date"] <= WINDOW_END)].copy()
    print(f"  After window filter [{WINDOW_START.date()} → {WINDOW_END.date()}]: {len(df):,}")

    # Clip spans to the window.
    df["start"] = df["permit_start_date"].clip(lower=WINDOW_START)
    df["end"]   = df["permit_end_date"].clip(upper=WINDOW_END)
    # Sanity: drop inverted spans.
    df = df[df["start"] <= df["end"]].copy()

    print(f"  Permit-type distribution:")
    print(df["permit_type"].value_counts().to_string())
    return df[["cnn", "permit_type", "start", "end"]]

## `expand_to_daily()`

Each permit row covers `[start, end]`. Expand to one row per (cnn, date) so the feature table joins row-for-row against the daily training grid. `pd.date_range` per permit is O(n · span); with ~2k blocking permits * average ~30 day span we stay under 100k rows.

In [4]:
def expand_to_daily(permits):
    rows = []
    for _, p in permits.iterrows():
        dates = pd.date_range(p["start"].normalize(),
                             p["end"].normalize(), freq="D")
        for d in dates:
            rows.append((p["cnn"], d.date()))
    daily = pd.DataFrame(rows, columns=["cnn", "date"])
    # Collapse duplicates (multiple overlapping permits on the same block
    # should still just flag as permit_active=1 for the day).
    daily = daily.drop_duplicates()
    daily["permit_active"] = 1
    print(f"  (cnn, date) permit-day rows: {len(daily):,}")
    print(f"  Unique blocks with ≥1 blocking-permit day: {daily['cnn'].nunique():,}")
    return daily

## `add_rolling_count()`

30-day rolling count of permit-active days per block. Signals chronic disruption (a block under a multi-month construction project) vs. a one-day emergency repair — the former depresses occupancy for weeks after the permit closes while the block is restriped / crews return.

In [5]:
def add_rolling_count(daily):
    # Build a full (cnn, date) grid over the window for cnn's that ever had a
    # permit, so the rolling window sees zeros on off-days.
    all_dates = pd.date_range(WINDOW_START.normalize(),
                             WINDOW_END.normalize(), freq="D").date
    cnns = daily["cnn"].unique()
    grid = pd.MultiIndex.from_product([cnns, all_dates],
                                      names=["cnn", "date"]).to_frame(index=False)
    grid = grid.merge(daily, on=["cnn", "date"], how="left").fillna({"permit_active": 0})

    grid = grid.sort_values(["cnn", "date"])
    grid["permit_count_30d"] = (grid.groupby("cnn")["permit_active"]
                                    .rolling(window=30, min_periods=1)
                                    .sum().reset_index(level=0, drop=True)
                                    .astype(int))
    # Keep only days where something is interesting (active today OR recent).
    grid = grid[(grid["permit_active"] == 1) | (grid["permit_count_30d"] > 0)].copy()
    grid["permit_active"] = grid["permit_active"].astype(int)
    print(f"  Final permit feature rows: {len(grid):,}")
    return grid[["cnn", "date", "permit_active", "permit_count_30d"]]

## Pipeline

In [6]:
print("=" * 60)
print("ParkCast SF — Street-Use Permit Features")
print("=" * 60)

permits = load_permits()
daily = expand_to_daily(permits)
feats = add_rolling_count(daily)

feats.to_csv(OUT, index=False)
print(f"\nSummary:")
print(feats[["permit_active", "permit_count_30d"]].describe().round(2).to_string())
print(f"\nSaved → {OUT}")
print("=" * 60)

ParkCast SF — Street-Use Permit Features
  Raw permits: 13,472
  After blocking-type filter: 2,563  (dropped 455)
  After window filter [2025-03-01 → 2026-05-01]: 2,226
  Permit-type distribution:
permit_type
Excavation    1577
TempOccup      535
ExcStreet       73


  (cnn, date) permit-day rows: 97,350
  Unique blocks with ≥1 blocking-permit day: 1,477
  Final permit feature rows: 120,745

Summary:
       permit_active  permit_count_30d
count      120745.00         120745.00
mean            0.81             22.62
std             0.40             10.37
min             0.00              1.00
25%             1.00             14.00
50%             1.00             30.00
75%             1.00             30.00
max             1.00             30.00

Saved → /Users/kayvan/Desktop/MSDS/ml-ops/Project/ml-ops-final-project-team-ParkCast-SF/data/block_permit_features.csv
